# Food Desert Case Study

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/07-food-desert-case-study.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

- Apply USDA food desert criteria using SocialMapper
- Analyze food access patterns and demographic characteristics
- Create food desert risk scores for multiple locations
- Visualize food access disparities
- Understand methodology limitations and data quality caveats

## Prerequisites

- Completed [01-Getting Started](01-getting-started.ipynb)
- Completed [02-Isochrone Analysis](02-isochrone-analysis.ipynb)
- Completed [03-Points of Interest](03-points-of-interest.ipynb)
- Completed [04-Census Data](04-census-data.ipynb)

## Background: What is a Food Desert?

A **food desert** is a geographic area where residents have limited access to affordable, nutritious food—particularly fresh fruits, vegetables, and other healthful whole foods.

### USDA Food Access Research Atlas Criteria

The USDA Economic Research Service (ERS) defines food deserts using specific distance and income thresholds:

| Area Type | Distance Threshold | Income Criteria |
|-----------|-------------------|----------------|
| **Urban** | > 1 mile (1.6 km) to supermarket | Poverty rate ≥ 20% OR median family income ≤ 80% of area median |
| **Rural** | > 10 miles (16 km) to supermarket | Same as above |

> **Source:** USDA Economic Research Service, [Food Access Research Atlas](https://www.ers.usda.gov/data-products/food-access-research-atlas/documentation/), 2023.

### Why These Thresholds?

**1-mile urban threshold:**
- Represents approximately 15-20 minutes of walking
- Accounts for households without reliable vehicle access
- Based on research showing walking as primary mode for low-income urban food shopping

**10-mile rural threshold:**
- Acknowledges car-dependent nature of rural areas
- Represents approximately 15-20 minutes of driving
- Based on research on rural food shopping patterns

> **Note:** The USDA also provides alternative thresholds (0.5 miles, 10 miles, 20 miles) for more nuanced analysis. This notebook focuses on the standard 1-mile urban threshold.

## Setup

In [ ]:
# Install SocialMapper (pin versions for Colab compatibility)
!pip install -q "socialmapper[routing]" "pandas<3.0" "numpy<2.0" folium

# IMPORTANT: After install, go to Runtime > Restart session, then skip this cell

In [ ]:
import os
import json
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

import socialmapper
print(f"SocialMapper v{socialmapper.__version__}")

from socialmapper import (
    create_isochrone,
    get_poi,
    get_census_blocks,
    get_census_data,
    create_map,
    SocialMapperError,
    ValidationError
)
from shapely.geometry import shape, Point
from IPython.display import Image, display
import folium

print("Ready for food desert analysis!")

## Methodology Overview

Our analysis follows these steps:

1. **Define access area**: Create 15-minute walking isochrone (~1 mile/1.6 km)
2. **Find food sources**: Query grocery stores and supermarkets from OpenStreetMap
3. **Analyze demographics**: Get census population and income data
4. **Apply criteria**: Classify areas based on USDA-inspired thresholds
5. **Visualize results**: Create maps showing food access patterns

> **Warning:** This analysis is for educational purposes. Official USDA food desert designations use proprietary data sources and more sophisticated methodology. See [Methodology Limitations](#methodology-limitations) section.

## Part 1: Single Location Analysis

Let's start by analyzing food access for a specific location.

In [ ]:
# Define study location
location = "Detroit, MI"

print(f"Food Access Analysis: {location}")
print("=" * 50)

### Step 1: Define Walking Distance

We use a 15-minute walk as our accessibility threshold, which approximates the USDA's 1-mile standard.

| Walking Speed | Time | Distance |
|--------------|------|----------|
| Average (3.1 mph) | 15 min | ~0.78 miles (1.25 km) |
| Slow (2.5 mph) | 15 min | ~0.63 miles (1.0 km) |
| Fast (3.5 mph) | 15 min | ~0.88 miles (1.4 km) |

In [ ]:
# Create walking isochrone (15 minutes ≈ 1 mile)
walk_area = create_isochrone(
    location=location,
    travel_time=15,
    travel_mode="walk"
)

print(f"Walking area (15 min): {walk_area['properties']['area_sq_km']:.2f} km²")
print(f"Approximate radius: ~{walk_area['properties']['area_sq_km']**0.5:.2f} km")

### Step 2: Find Food Sources

> **Data Quality Note:** POI data comes from OpenStreetMap, which has variable coverage. Urban areas typically have better coverage than rural areas. Some stores may be missing or incorrectly categorized.

In [ ]:
# Find grocery stores and supermarkets
# Note: The 'shopping' category includes supermarkets, convenience stores, etc.
food_sources = get_poi(
    location=location,
    categories=["shopping"],
    travel_time=15,
    limit=50
)

print(f"Food sources within 15-min walk: {len(food_sources)}")

if food_sources:
    print("\nNearest stores:")
    for store in sorted(food_sources, key=lambda x: x['distance_km'])[:5]:
        print(f"  - {store['name']}: {store['distance_km']:.2f} km")
else:
    print("\nWARNING: No grocery stores within walking distance!")
    print("This area may be a food desert.")

### Step 3: Analyze Demographics

We retrieve census data to understand the population characteristics of the study area.

In [ ]:
# Robust data aggregation functions (handles None and negative values)
def safe_value(value, default=0):
    """Return value if valid, otherwise default."""
    if value is None or (isinstance(value, (int, float)) and value < 0):
        return default
    return value

def safe_average(values):
    """Calculate average of valid values only."""
    valid = [v for v in values if v is not None and v > 0]
    return sum(valid) // len(valid) if valid else 0

In [ ]:
# Get census blocks and demographics
blocks = get_census_blocks(polygon=walk_area)
geoids = [b['geoid'] for b in blocks]

census_result = get_census_data(
    location=geoids,
    variables=["population", "median_income", "total_households"]
)

# Combine data with robust handling
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    block['population'] = safe_value(data.get('population', 0))
    block['median_income'] = safe_value(data.get('median_income', 0))
    block['households'] = safe_value(data.get('total_households', 0))

# Calculate totals
total_pop = sum(b['population'] for b in blocks)
total_households = sum(b['households'] for b in blocks)
incomes = [b['median_income'] for b in blocks if b['median_income'] > 0]

print(f"\nDemographic Profile:")
print(f"  Census blocks: {len(blocks)}")
print(f"  Population affected: {total_pop:,}")
print(f"  Households: {total_households:,}")
if incomes:
    print(f"  Average median income: ${safe_average(incomes):,}")
else:
    print(f"  Average median income: Data unavailable")

### Step 4: Food Desert Classification

We apply criteria inspired by the USDA definition:

| Classification | Criteria |
|---------------|----------|
| **FOOD DESERT** | Low income AND low access (< 2 stores within 15-min walk) |
| **LOW ACCESS** | Limited food access (< 2 stores) but adequate income |
| **LOW INCOME** | Limited income but adequate food access |
| **ADEQUATE** | Good income and food access |

> **Note:** Our income threshold ($50,000) is simplified. The USDA uses tract-level poverty rates and comparison to area median income.

In [ ]:
def classify_food_access(num_stores, income, is_urban=True):
    """
    Classify food access based on USDA-inspired criteria.
    
    Parameters:
        num_stores: Number of grocery stores within walking distance
        income: Average median household income for the area
        is_urban: Whether the area is urban (affects distance threshold)
    
    Returns:
        Tuple of (classification, reason)
    """
    # Low income threshold (simplified from USDA's 80% of area median)
    # National median household income ~$75,000 (2023), 80% = ~$60,000
    # We use $50,000 as a conservative threshold
    low_income = income < 50000
    
    # Low access: fewer than 2 stores within walking distance
    # Rationale: Having only 1 store creates vulnerability if it closes
    low_access = num_stores < 2
    
    if low_income and low_access:
        return "FOOD DESERT", "Low income AND low access"
    elif low_access:
        return "LOW ACCESS", "Limited food access"
    elif low_income:
        return "LOW INCOME", "Limited income but adequate access"
    else:
        return "ADEQUATE", "Good income and food access"

In [ ]:
# Classify this area
avg_income = safe_average(incomes)
classification, reason = classify_food_access(len(food_sources), avg_income)

print(f"\n{'='*50}")
print(f"FOOD ACCESS CLASSIFICATION: {classification}")
print(f"{'='*50}")
print(f"Reason: {reason}")
print(f"  - Grocery stores: {len(food_sources)}")
print(f"  - Avg median income: ${avg_income:,}")
print(f"  - Population affected: {total_pop:,}")

## Part 2: Multi-Area Comparison

Compare food access across different neighborhoods to identify patterns and disparities.

In [ ]:
def analyze_food_access(location):
    """Analyze food access for a location.
    
    Returns a dictionary with food access metrics.
    """
    try:
        # Get walkable area
        walk_area = create_isochrone(
            location=location,
            travel_time=15,
            travel_mode="walk"
        )
        
        # Find food sources
        food_sources = get_poi(
            location=location,
            categories=["shopping"],
            travel_time=15,
            limit=50
        )
        
        # Get demographics
        blocks = get_census_blocks(polygon=walk_area)
        geoids = [b['geoid'] for b in blocks]
        census = get_census_data(geoids, variables=["population", "median_income"])
        
        # Calculate metrics with robust handling
        total_pop = sum(
            safe_value(census.data.get(g, {}).get('population', 0))
            for g in geoids
        )
        incomes = [
            census.data.get(g, {}).get('median_income', 0)
            for g in geoids
            if safe_value(census.data.get(g, {}).get('median_income', 0)) > 0
        ]
        avg_income = safe_average(incomes)
        
        # Classify
        classification, _ = classify_food_access(len(food_sources), avg_income)
        
        return {
            "location": location,
            "food_stores": len(food_sources),
            "population": total_pop,
            "avg_income": avg_income,
            "classification": classification,
            "area_sq_km": walk_area['properties']['area_sq_km'],
            "error": None
        }
    except SocialMapperError as e:
        return {
            "location": location,
            "error": str(e)
        }

In [ ]:
# Compare multiple neighborhoods
neighborhoods = [
    "Downtown Detroit, MI",
    "Midtown Detroit, MI",
    "Corktown, Detroit, MI",
    "Hamtramck, MI"
]

print("Neighborhood Food Access Comparison")
print("=" * 70)

results = []
for hood in neighborhoods:
    result = analyze_food_access(hood)
    
    if result.get('error'):
        print(f"\n{hood}: Error - {result['error']}")
        continue
        
    results.append(result)
    
    print(f"\n{result['location']}:")
    print(f"  Food stores: {result['food_stores']}")
    print(f"  Population: {result['population']:,}")
    print(f"  Avg Income: ${result['avg_income']:,}")
    print(f"  Status: {result['classification']}")

In [ ]:
# Summary statistics
if results:
    food_deserts = [r for r in results if r['classification'] == "FOOD DESERT"]
    low_access = [r for r in results if r['classification'] in ["FOOD DESERT", "LOW ACCESS"]]

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"Total areas analyzed: {len(results)}")
    print(f"Food deserts: {len(food_deserts)}")
    print(f"Low access areas: {len(low_access)}")

    # Population in food deserts
    pop_in_deserts = sum(r['population'] for r in food_deserts)
    total_pop = sum(r['population'] for r in results)

    if total_pop > 0:
        print(f"\nPopulation in food deserts: {pop_in_deserts:,} ({pop_in_deserts/total_pop*100:.1f}%)")
        print(f"Total population analyzed: {total_pop:,}")

## Part 3: Visualization

In [ ]:
# Create a comprehensive food access map
location = "Detroit, MI"

# Get larger study area
study_area = create_isochrone(location, travel_time=30, travel_mode="drive")
blocks = get_census_blocks(polygon=study_area)

# Get census data
geoids = [b['geoid'] for b in blocks]
census = get_census_data(geoids, variables=["population", "median_income"])

# Add data to blocks with robust handling
for block in blocks:
    data = census.data.get(block['geoid'], {})
    block['population'] = safe_value(data.get('population', 0))
    block['median_income'] = safe_value(data.get('median_income', 0))

# Create income map
valid_blocks = [b for b in blocks if b['median_income'] > 0]

if valid_blocks:
    income_map = create_map(
        data=valid_blocks,
        column="median_income",
        title="Median Income - Detroit Metro Area",
        cmap="RdYlGn",
        show_stats=True,
        save_path="detroit_income.png"
    )

    display(Image(filename="detroit_income.png"))
else:
    print("No valid census blocks with income data available.")

### Interactive Food Access Map

In [ ]:
# Create interactive map with grocery stores
center = (42.3314, -83.0458)  # Detroit
m = folium.Map(location=center, zoom_start=12)

# Get grocery stores for larger area
all_groceries = get_poi(
    location=center,
    categories=["shopping"],
    limit=100
)

# Add grocery store markers
for store in all_groceries:
    folium.Marker(
        location=[store['lat'], store['lon']],
        popup=f"{store['name']}<br>Distance: {store['distance_km']:.2f} km",
        icon=folium.Icon(color='green', icon='shopping-cart', prefix='fa')
    ).add_to(m)

# Add census blocks colored by income
for block in valid_blocks[:50]:  # Limit for performance
    income = block['median_income']
    
    # Color by income level
    if income < 30000:
        color = '#d73027'  # Red - low income
    elif income < 50000:
        color = '#fc8d59'  # Orange
    elif income < 75000:
        color = '#fee08b'  # Yellow
    else:
        color = '#91cf60'  # Green - high income
    
    folium.GeoJson(
        block['geometry'],
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': 'gray',
            'weight': 0.5,
            'fillOpacity': 0.5
        },
        tooltip=f"Income: ${income:,}<br>Pop: {block['population']:,}"
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000;
            background-color: white; padding: 10px; border-radius: 5px;
            border: 2px solid gray;">
    <b>Legend</b><br>
    <i style="color: green">●</i> Grocery Store<br>
    <div style="background: #d73027; width: 20px; height: 10px; display: inline-block;"></div> Income < $30k<br>
    <div style="background: #fc8d59; width: 20px; height: 10px; display: inline-block;"></div> Income $30-50k<br>
    <div style="background: #fee08b; width: 20px; height: 10px; display: inline-block;"></div> Income $50-75k<br>
    <div style="background: #91cf60; width: 20px; height: 10px; display: inline-block;"></div> Income > $75k
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f"Map shows {len(all_groceries)} grocery stores")
m

## Part 4: Food Desert Score

We create a composite score (0-100) to quantify food desert risk. Higher scores indicate worse food access.

| Component | Points | Rationale |
|-----------|--------|----------|
| Store access | 0-40 | Primary indicator of food availability |
| Income level | 0-30 | Economic barrier to food access |
| Store density | 0-30 | Competition and choice availability |

In [ ]:
def calculate_food_desert_score(location):
    """
    Calculate a food desert score (0-100) for a location.
    
    Higher scores indicate worse food access.
    
    Components:
    - Store access (0-40 points): Number of stores within walking distance
    - Income level (0-30 points): Area median income level
    - Store density (0-30 points): Stores per square kilometer
    
    Returns:
        Dictionary with score, risk level, and component details
    """
    try:
        # Get data
        walk_area = create_isochrone(location, travel_time=15, travel_mode="walk")
        area_km = walk_area['properties']['area_sq_km']
        
        food_sources = get_poi(
            location=location,
            categories=["shopping"],
            travel_time=15,
            limit=50
        )
        
        blocks = get_census_blocks(polygon=walk_area)
        geoids = [b['geoid'] for b in blocks]
        census = get_census_data(geoids, variables=["population", "median_income"])
        
        # Calculate metrics with robust handling
        total_pop = sum(
            safe_value(census.data.get(g, {}).get('population', 0))
            for g in geoids
        )
        incomes = [
            census.data.get(g, {}).get('median_income', 0)
            for g in geoids
            if safe_value(census.data.get(g, {}).get('median_income', 0)) > 0
        ]
        avg_income = safe_average(incomes)
        
        # Score components
        score = 0
        details = []
        
        # Store access (0-40 points, more stores = lower score)
        if len(food_sources) == 0:
            store_score = 40
        elif len(food_sources) == 1:
            store_score = 30
        elif len(food_sources) <= 3:
            store_score = 15
        else:
            store_score = 0
        score += store_score
        details.append(f"Store access: {store_score}/40")
        
        # Income level (0-30 points, lower income = higher score)
        if avg_income < 25000:
            income_score = 30
        elif avg_income < 40000:
            income_score = 20
        elif avg_income < 60000:
            income_score = 10
        else:
            income_score = 0
        score += income_score
        details.append(f"Income level: {income_score}/30")
        
        # Store density (0-30 points, fewer stores per km² = higher score)
        density = len(food_sources) / area_km if area_km > 0 else 0
        if density < 0.5:
            density_score = 30
        elif density < 1.0:
            density_score = 20
        elif density < 2.0:
            density_score = 10
        else:
            density_score = 0
        score += density_score
        details.append(f"Store density: {density_score}/30")
        
        # Classify risk level
        if score >= 70:
            risk = "CRITICAL"
        elif score >= 50:
            risk = "HIGH"
        elif score >= 30:
            risk = "MODERATE"
        else:
            risk = "LOW"
        
        return {
            "location": location,
            "score": score,
            "risk": risk,
            "details": details,
            "food_stores": len(food_sources),
            "population": total_pop,
            "avg_income": avg_income,
            "area_km": area_km,
            "error": None
        }
    except SocialMapperError as e:
        return {
            "location": location,
            "error": str(e)
        }

In [ ]:
# Calculate scores for multiple areas
test_locations = [
    "Detroit, MI",
    "Manhattan, NY",
    "Beverly Hills, CA",
    "South Side Chicago, IL"
]

print("Food Desert Risk Assessment")
print("=" * 70)

all_scores = []
for loc in test_locations:
    result = calculate_food_desert_score(loc)
    
    if result.get('error'):
        print(f"\n{loc}: Error - {result['error']}")
        continue
        
    all_scores.append(result)
    
    print(f"\n{result['location']}")
    print(f"  Score: {result['score']}/100 - {result['risk']} RISK")
    for detail in result['details']:
        print(f"    {detail}")
    print(f"  Food stores: {result['food_stores']}, Population: {result['population']:,}")

## Part 5: Recommendations

In [ ]:
def generate_recommendations(score_result):
    """Generate policy and intervention recommendations based on food desert analysis."""
    
    recommendations = []
    
    if score_result.get('food_stores', 0) == 0:
        recommendations.append("URGENT: No grocery stores within walking distance")
        recommendations.append("  - Consider mobile grocery programs")
        recommendations.append("  - Evaluate locations for new grocery development")
        recommendations.append("  - Implement community garden initiatives")
    
    elif score_result.get('food_stores', 0) < 3:
        recommendations.append("Limited food access detected")
        recommendations.append("  - Support existing grocery stores with incentives")
        recommendations.append("  - Explore farmers market opportunities")
    
    if score_result.get('avg_income', 0) < 40000:
        recommendations.append("Low-income area identified")
        recommendations.append("  - Expand SNAP retailer participation")
        recommendations.append("  - Consider food assistance programs")
        recommendations.append("  - Partner with food banks for distribution")
    
    if score_result.get('score', 0) >= 50:
        recommendations.append("High-priority area for intervention")
        recommendations.append("  - Conduct detailed community needs assessment")
        recommendations.append("  - Engage local stakeholders for solutions")
    
    if not recommendations:
        recommendations.append("Area has adequate food access")
        recommendations.append("  - Monitor for changes in store availability")
        recommendations.append("  - Maintain support for healthy food initiatives")
    
    return recommendations

# Generate recommendations for each area
if all_scores:
    print("\n" + "=" * 70)
    print("RECOMMENDATIONS")
    print("=" * 70)

    for result in all_scores:
        print(f"\n{result['location']} ({result['risk']} Risk):")
        recs = generate_recommendations(result)
        for rec in recs:
            print(f"  {rec}")

## Part 6: Export Report

In [ ]:
# Create comprehensive report
from datetime import datetime

if all_scores:
    report = {
        "title": "Food Desert Analysis Report",
        "date": datetime.now().strftime("%Y-%m-%d"),
        "methodology": {
            "walking_threshold": "15 minutes (~1 mile)",
            "income_threshold": "$50,000 median household income",
            "data_sources": [
                "OpenStreetMap (POI data)",
                "US Census Bureau (demographics)"
            ],
            "limitations": [
                "OSM data coverage varies by location",
                "Simplified income thresholds vs USDA methodology",
                "Does not account for store quality or pricing",
                "Walking isochrones may not reflect actual barriers"
            ]
        },
        "summary": {
            "areas_analyzed": len(all_scores),
            "high_risk_areas": len([s for s in all_scores if s['risk'] in ['HIGH', 'CRITICAL']]),
            "total_population_at_risk": sum(
                s['population'] for s in all_scores 
                if s['risk'] in ['HIGH', 'CRITICAL']
            )
        },
        "areas": [
            {
                "location": s['location'],
                "score": s['score'],
                "risk_level": s['risk'],
                "food_stores": s['food_stores'],
                "population": s['population'],
                "avg_income": s['avg_income'],
                "recommendations": generate_recommendations(s)
            }
            for s in all_scores
        ]
    }

    # Save report
    with open("food_desert_report.json", "w") as f:
        json.dump(report, f, indent=2)

    print("Report saved: food_desert_report.json")
    print("\nReport Summary:")
    print(f"  Areas analyzed: {report['summary']['areas_analyzed']}")
    print(f"  High-risk areas: {report['summary']['high_risk_areas']}")
    print(f"  Population at risk: {report['summary']['total_population_at_risk']:,}")

## Methodology Limitations

This analysis has several important limitations that should be considered when interpreting results:

### Data Quality Issues

| Issue | Impact | Mitigation |
|-------|--------|------------|
| **OSM coverage varies** | Some stores may be missing | Cross-reference with local data sources |
| **Store categorization** | Convenience stores may be included/excluded | Manual verification for critical decisions |
| **Census data currency** | Data may be 1-5 years old | Check ACS data release year |
| **Boundary effects** | Areas near study boundary may be underserved | Extend analysis area slightly |

### Methodological Simplifications

1. **Income threshold**: We use a single $50,000 threshold, while USDA uses:
   - Poverty rate ≥ 20%, OR
   - Median family income ≤ 80% of state/metro median

2. **Distance measurement**: Walking isochrones account for road networks but not:
   - Physical barriers (stairs, steep hills)
   - Safety concerns (crime, traffic)
   - Weather and seasonal accessibility

3. **Store quality not assessed**: We count stores but don't evaluate:
   - Selection and availability of healthy foods
   - Pricing and affordability
   - Store hours and accessibility

4. **Vehicle access not modeled**: Rural food desert analysis requires:
   - 10-mile driving threshold
   - Vehicle ownership rates
   - Public transit availability

> **Warning:** This analysis is for educational and exploratory purposes. Official food desert designations should use USDA Food Access Research Atlas data and methodology.

## How to Cite

### Citing This Analysis

If you use this methodology or SocialMapper in academic work:

```
SocialMapper (2025). Food Desert Analysis Tutorial. 
Available at: https://github.com/mihiarc/socialmapper
```

### Official Data Sources to Reference

For food desert research, cite the authoritative USDA sources:

**USDA Food Access Research Atlas:**
> Economic Research Service (ERS), U.S. Department of Agriculture. (2023). 
> Food Access Research Atlas. 
> Available at: https://www.ers.usda.gov/data-products/food-access-research-atlas/

**USDA Food Desert Methodology:**
> Ver Ploeg, M., Breneman, V., Farrigan, T., et al. (2009). 
> Access to Affordable and Nutritious Food: Measuring and Understanding 
> Food Deserts and Their Consequences. 
> U.S. Department of Agriculture, Economic Research Service, Report to Congress.
> Available at: https://www.ers.usda.gov/publications/pub-details/?pubid=42729

**Census Data:**
> U.S. Census Bureau. American Community Survey 5-Year Estimates.
> Available at: https://data.census.gov/

**OpenStreetMap Data:**
> OpenStreetMap contributors. (2025). OpenStreetMap.
> Available at: https://www.openstreetmap.org/
> (Data is available under the Open Database License)

## Troubleshooting

### Common Issues and Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| No grocery stores found | OSM coverage gap | Try larger search radius, verify on map |
| Census data missing | Geographic mismatch | Check if location is in US |
| Slow isochrone generation | NetworkX backend | Use `valhalla` backend instead |
| Income data shows 0 | Census suppression | Use tract-level data instead of block |
| Inconsistent results | Different backends | Specify backend explicitly |

In [ ]:
# Example: Debug missing data
def debug_food_access(location):
    """Debug function to diagnose food access analysis issues."""
    print(f"Debugging: {location}")
    print("=" * 50)
    
    # Check isochrone generation
    try:
        iso = create_isochrone(location, travel_time=15, travel_mode="walk")
        print(f"✓ Isochrone created: {iso['properties']['area_sq_km']:.2f} km²")
    except Exception as e:
        print(f"✗ Isochrone failed: {e}")
        return
    
    # Check POI query
    try:
        pois = get_poi(location, categories=["shopping"], limit=20)
        print(f"✓ POI query: {len(pois)} stores found")
        if len(pois) == 0:
            print("  Note: Try 'food_and_drink' category or increase limit")
    except Exception as e:
        print(f"✗ POI query failed: {e}")
    
    # Check census blocks
    try:
        blocks = get_census_blocks(polygon=iso)
        print(f"✓ Census blocks: {len(blocks)} blocks found")
    except Exception as e:
        print(f"✗ Census blocks failed: {e}")
        return
    
    # Check census data
    try:
        geoids = [b['geoid'] for b in blocks]
        census = get_census_data(geoids, variables=["population", "median_income"])
        
        # Count data availability
        has_pop = sum(1 for g in geoids if census.data.get(g, {}).get('population'))
        has_income = sum(1 for g in geoids if census.data.get(g, {}).get('median_income'))
        
        print(f"✓ Census data: {has_pop}/{len(geoids)} with population, {has_income}/{len(geoids)} with income")
        if has_income == 0:
            print("  Note: Income data may be suppressed at block level")
    except Exception as e:
        print(f"✗ Census data failed: {e}")

# Run debug
debug_food_access("Detroit, MI")

## Conclusion

This case study demonstrated how to use SocialMapper for food access equity analysis:

1. **Isochrones** define walkable access areas (~1 mile / 15-min walk)
2. **POI queries** find food sources from OpenStreetMap
3. **Census data** provides demographic context
4. **Scoring systems** quantify food desert risk
5. **Visualization** communicates findings effectively

### Key Insights

- Food deserts disproportionately affect low-income communities
- Walking distance is critical for car-free households
- Solutions require understanding both access AND affordability
- Data quality varies significantly by location

### Next Steps

- Expand analysis to entire cities using `analyze_multiple_pois()`
- Include convenience stores and farmers markets
- Track changes over time with historical data
- Integrate with official USDA Food Access Research Atlas data
- Generate HTML reports using `generate_report()`